# About

## Contents

- [Sequence-informed prediction of microbial growth temperature-performance curves](#sequence-informed-prediction-of-microbial-growth-temperature-performance-curves)
- [What this toolkit can be used for](#what-this-toolkit-can-be-used-for)
- [Overview of the model architecture](#overview-of-the-model-architecture)
- [Model components](#model-components)
- [Current implementation](#current-implementation)
- [How to use the toolkit](#how-to-use-the-toolkit)
- [Output summary](#output-summary)
- [Future development](#future-development)


# About

## Sequence-informed prediction of microbial growth temperature-performance curves

The **Microbial Growth TPC Predictor** is a modelling toolkit for predicting microbial growth temperature-performance curves (TPCs) from genome- and proteome-level sequence information. The aim of the toolkit is to predict the relative or absolute growth performance of microorganisms across temperature when complete experimental growth curves are unavailable.

Microbial growth is strongly temperature dependent. Different species may differ in their optimal growth temperature, low-temperature tolerance, high-temperature sensitivity, and the shape of growth decline beyond the thermal optimum. Therefore, predicting a single optimal growth temperature is not sufficient to describe microbial thermal adaptation. This toolkit aims to predict the complete growth temperature-performance curve from sequence information, thereby providing a more systematic description of how microbial growth rate varies with temperature.

The current version mainly focuses on bacteria and archaea. It predicts normalised microbial growth TPCs and can optionally use flux balance analysis (FBA) to estimate a peak growth rate, allowing the normalised curve to be converted into an absolute growth-rate TPC.


## What this toolkit can be used for

This toolkit is designed for questions in microbial ecology, thermal adaptation, genome-informed trait prediction, and physiological modelling. It is particularly useful for cross-species prediction when experimental temperature-response data are limited.

Typical applications include:

- predicting complete microbial growth temperature-performance curves from genome or proteome information;
- predicting the optimal growth temperature (OGT) of bacterial and archaeal species;
- comparing thermal adaptation strategies across psychrophilic, mesophilic, thermophilic, and archaeal taxa;
- generating species-specific normalised temperature-response functions;
- using FBA to scale normalised TPCs into absolute growth-rate curves when genome or metabolic information is available;
- providing temperature-dependent growth traits for downstream ecological models, community simulations, or environmental response analyses.


## Overview of the model architecture

The toolkit consists of three connected modelling layers. The first layer predicts OGT from ESM-2 proteome embeddings. The second layer predicts the shape of the microbial growth TPC from proteome embeddings. The third layer optionally uses FBA as an absolute-scale anchor to convert the normalised TPC into a growth-rate curve with physical units.

The first layer uses sequence-derived proteome representations to predict the optimal growth temperature of the target microorganism. Specifically, protein sequences are represented using ESM-2 embeddings, and the resulting organism-level proteome embedding is used to infer OGT. This predicted OGT is subsequently used as a biological anchor for the peak position of the TPC.

The second layer uses the mean-pooled ESM-2 proteome embedding to represent organism-level protein sequence information and passes it into the core TPC shape prediction model. This model combines neural network components, a mechanism-inspired Universal Temperature Performance Curve (UTPC) structure, and a constrained residual correction module to predict a complete growth TPC whose peak is normalised to 1.

The third layer is an optional metabolic anchoring step. When the user requires an absolute growth-rate curve rather than only a relative growth response, genome-scale metabolic reconstruction and FBA can be used to estimate the peak growth rate under a specified medium condition. The model then scales the normalised TPC by this estimated peak growth rate, producing a growth-rate TPC with absolute units.

Overall, the model can be summarised as:

```text
genome / proteome information
→ ESM-2 proteome embedding
→ OGT prediction
→ proteome embedding-based TPC shape prediction
→ normalised growth TPC
→ optional FBA-based absolute scaling
→ absolute growth-rate TPC
```


## Model components

### 1. OGT prediction module

The OGT prediction module estimates the optimal growth temperature of a microorganism from proteome-level sequence representations. Protein sequences are first encoded using the ESM-2 protein language model. The resulting protein-level embeddings are then aggregated into an organism-level proteome embedding, which is used as the input to the OGT prediction model.

The model outputs a predicted OGT in degrees Celsius. This predicted value is not only an independent thermal adaptation trait, but also serves as a constraint on the peak temperature of the downstream TPC prediction model.

### 2. TPC shape prediction module

The TPC shape prediction module is the core component of the toolkit. It uses the ESM-2 protein language model to obtain protein sequence representations, which are then aggregated by mean pooling to generate an organism-level proteome embedding.

This embedding is passed into the core neural network model to predict the shape parameters of the TPC. The model includes a mechanism-inspired component based on the Universal Temperature Performance Curve (UTPC), which provides a biologically meaningful baseline curve. In addition, a residual correction module is used to capture species-specific deviations that may not be fully represented by the UTPC component alone.

To reduce biologically implausible predictions, the model includes constraints on curve behaviour, with particular emphasis on the post-peak decline. The final output is a normalised growth TPC in which the maximum growth value is scaled to 1.

### 3. FBA anchor module

The FBA anchor module is an optional component for converting the normalised TPC into an absolute growth-rate curve. This module can construct a genome-scale metabolic model from the input genome and run flux balance analysis under user-defined medium conditions.

The peak growth rate obtained from FBA is treated as an absolute-scale anchor. The model then multiplies the normalised TPC by this peak growth rate to obtain absolute growth-rate predictions across temperature.

If the user does not run FBA, the toolkit still outputs a normalised TPC. This normalised curve can still be used to compare the shape of thermal responses across species.


## Current implementation

The toolkit is available in both modular and consolidated single-file forms.

The modular implementation mainly includes:

- `OGT_predictor.py`: used for training and applying the OGT prediction model;
- `core_model.py`: used for training the core TPC shape prediction model;
- `TPC_predictor.py`: used for predicting TPCs from trained models;
- `FBA_anchor_point.py`: used for optional FBA-based estimation of peak growth rate.

For easier execution and documentation, the main functions have also been consolidated into a single script:

```text
microbial_tpc_single_file.py
```

This file provides a unified command-line interface for OGT prediction, TPC prediction, and FBA anchoring.


## How to use the toolkit

The toolkit is designed to predict a complete microbial growth temperature-performance curve from a user-provided sequence file and medium condition. The output is a growth-rate TPC with physical units over a user-defined temperature range.

The user needs to provide three inputs:

1. a FASTA file for the target microorganism;
2. a medium definition file;
3. the temperature range over which growth should be predicted.

The final output is a CSV file containing the predicted absolute growth rate at each temperature.

### 1. Prepare the FASTA file

The user first prepares a FASTA file for the target microorganism, for example:

```text
genome.fna
```

This file can be a genome FASTA file for the target microorganism. The toolkit uses this sequence information to generate the representations needed for OGT prediction, TPC shape prediction, and metabolic model construction.

### 2. Prepare the medium file

The user also needs to prepare a medium file defining the nutrient conditions for the FBA model. The medium file uses JSON format, for example:

```json
{
  "EX_glc__D_e": 10.0,
  "EX_o2_e": 20.0,
  "EX_nh4_e": 1000.0,
  "EX_pi_e": 1000.0,
  "EX_so4_e": 1000.0,
  "EX_h2o_e": 1000.0,
  "EX_h_e": 1000.0
}
```

Each entry specifies an exchange reaction that is allowed to take up a nutrient, together with its uptake bound. For example, `EX_glc__D_e` denotes D-glucose, `EX_o2_e` denotes oxygen, and the numerical values specify the maximum available flux for each compound.

To modify the medium condition, the user only needs to edit this JSON file. For example, the glucose uptake bound can be changed from:

```json
"EX_glc__D_e": 10.0
```

to:

```json
"EX_glc__D_e": 2.0
```

to represent a lower-glucose environment.

To simulate an anaerobic condition, oxygen can be set to zero:

```json
"EX_o2_e": 0.0
```

The exchange reaction IDs in the medium file must match the reaction IDs in the metabolic model.

### 3. Set the temperature range

The user specifies the temperature range using the minimum temperature, maximum temperature, and temperature step size. For example:

```text
--temp_min 5
--temp_max 80
--temp_step 1
```

This setting predicts growth rate from 5°C to 80°C at 1°C intervals.

For a finer temperature grid, such as 0.5°C intervals, the user can specify:

```text
--temp_min 10
--temp_max 50
--temp_step 0.5
```

### 4. Run the full TPC prediction

After preparing the FASTA file, medium file, and temperature range, the user can run:

```bash
python microbial_tpc_single_file.py predict \
    --fasta genome.fna \
    --medium medium.json \
    --temp_min 5 \
    --temp_max 80 \
    --temp_step 1 \
    --output predicted_absolute_tpc.csv
```

This command performs the following steps:

1. generates sequence-based representations from the FASTA file;
2. predicts the optimal growth temperature of the target microorganism;
3. predicts the normalised TPC shape from proteome-level sequence information;
4. runs FBA under the user-defined medium condition to estimate the peak growth rate;
5. scales the normalised TPC using the FBA-derived peak growth rate;
6. outputs a complete growth-rate TPC with physical units.


## Output summary

The final output file is:

```text
predicted_absolute_tpc.csv
```

This file contains the predicted growth rate of the target microorganism across the specified temperature range. For example:

```text
temperature_C,relative_growth,absolute_growth_rate
5.0,0.01,0.004
6.0,0.02,0.008
7.0,0.03,0.012
...
37.0,1.00,0.400
...
80.0,0.00,0.000
```

Here, `temperature_C` is the prediction temperature in degrees Celsius; `relative_growth` is the normalised TPC value, with the peak scaled to 1; and `absolute_growth_rate` is the absolute growth rate after scaling by the FBA-derived peak growth rate, typically expressed in h⁻¹.

Thus, the user obtains a complete microbial growth-rate temperature-performance curve with physical units:

```text
temperature → absolute growth rate
```

This curve represents the predicted growth rate of the target microorganism across temperature under the specified medium condition.

The main outputs of the model include:

- predicted OGT;
- normalised relative growth values across temperature;
- TPC curve parameters;
- FBA-derived peak growth rate;
- absolute growth-rate TPC.

The normalised TPC can be written as:

```text
μ_relative(T) ∈ [0, 1]
```

When the FBA anchor is used, the absolute growth rate is calculated as:

```text
μ_absolute(T) = μ_relative(T) × μ_peak,FBA
```


## Future development

The current version focuses on sequence-informed prediction of bacterial and archaeal growth TPCs. Future development may extend the toolkit in several directions:

- improving proteome embedding strategies;
- incorporating more robust uncertainty estimation;
- adding phylogenetic constraints;
- extending the framework to broader taxonomic groups and additional physiological traits;
- improving the stability and automation of FBA anchoring;
- supporting batch prediction for multi-species datasets;
- validating extrapolation performance using additional experimental temperature-response datasets.

The long-term goal is to establish an interpretable predictive framework from genome or proteome sequence to microbial thermal physiology. This framework can provide a computational tool for studying microbial thermal adaptation and temperature-dependent growth when complete experimental TPC data are unavailable.
